In [1]:
import math
import numpy as np
import matplotlib
import torch
%matplotlib inline

In [2]:
# What are torch Tensors? What is requires_grad?? What is .double()? Just casting to float64, torch defaults to float32
# requires_grad defaults to False - you don't want grads for leaf nodes (data, bias)
x1 = torch.Tensor([2.0]).double()                ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()               ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()  ; b.requires_grad = True
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print('---')
print('x2', x2.grad.item())
print('w2', w2.grad.item())
print('x1', x1.grad.item())
print('w1', w1.grad.item())

0.7071066904050358
---
x2 0.5000001283844369
w2 0.0
x1 -1.5000003851533106
w1 1.0000002567688737


In [3]:
torch.Tensor([[1,2],[3,4],[5,6]]).shape

torch.Size([3, 2])

In [4]:
torch.Tensor([2.0])

tensor([2.])

In [5]:
# they print the same - implied not always, why?
o.item()
o.data.item()

0.7071066904050358

In [6]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __pow__(self, other):
    assert isinstance(other, (int, float)), "only supporting int/float powers for now"
    out = Value(self.data**other, (self,), f'**{other}')

    def _backward():
        self.grad += other * (self.data ** (other - 1)) * out.grad
    out._backward = _backward

    return out
  
  def __rmul__(self, other): # other * self
    return self * other

  def __truediv__(self, other): # self / other
    return self * other**-1

  def __neg__(self): # -self
    return self * -1

  def __sub__(self, other): # self - other
    return self + (-other)

  def __radd__(self, other): # other + self
    return self + other

  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self, ), 'exp')
    
    def _backward():
      self.grad += out.data * out.grad # NOTE: in the video I incorrectly used = instead of +=. Fixed here.
    out._backward = _backward
    
    return out
  
  
  def backward(self):
    
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [7]:
# reimplement Neurons using pytorch api
# nin - dimension of neuron/layer input
# What is below doing? Well I'd call MLP(4,2) to have 4D input and 2D output
class Neuron:
  
  def __init__(self, nin):
    self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
    self.b = Value(random.uniform(-1,1))
  
  def __call__(self, x):
    # w * x + b
    # Note we're forcing the sum to start at b
    act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
    out = act.tanh()
    return out
  
  def parameters(self):
    return self.w + [self.b]

# Layer is list of neurons
# nout - number of neurons
# nin - data dimension of neurons
class Layer:
  
  def __init__(self, nin, nout):
    self.neurons = [Neuron(nin) for _ in range(nout)]
  
  def __call__(self, x):
    outs = [n(x) for n in self.neurons]
    return outs[0] if len(outs) == 1 else outs
  
  def parameters(self):
    return [p for neuron in self.neurons for p in neuron.parameters()]

# MLP has size (total neurons), and layers (list of layers)
# nout - number of neurons in each layer beyond first, list
# nin - size of initial layer (doubles)
class MLP:
  
  def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
  
  def __call__(self, x):
    for layer in self.layers:
      x = layer(x)
    return x
  
  def parameters(self):
    return [p for layer in self.layers for p in layer.parameters()]

In [8]:
# Neuron with 2D input
import random
n = Neuron(2)
data = [1,2]
n(data)

Value(data=0.30718870382234115)

In [9]:
# Full MLP
net = MLP(3, [4,4,1])
net([2.0,3.0,-1.0])

Value(data=0.35658356834530547)

In [10]:
# Dataset example

xs = [
  [2.0, 3.0, -1.0],
  [3.0, -1.0, 0.5],
  [0.5, 1.0, 1.0],
  [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

In [11]:
n = MLP(3, [4,4,1])
for k in range(20):
  
  # forward pass - what is the current loss?
  ypred = [n(x) for x in xs]
  loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
  
  # backward pass - get all grads going right to left - remember the gradient msut be flushed back to zero (this is .zero_grad() in pytorch)
  for p in n.parameters():
    p.grad = 0.0
  loss.backward()
  
  # update - gradient descent
  for p in n.parameters():
    p.data += -0.1 * p.grad
  
  print(k, loss.data)

0 8.677509418842222
1 3.951078610617511
2 2.6478524530693788
3 0.6328407805593594
4 0.2582931349759288
5 0.12390501198674082
6 0.08477125728931996
7 0.0642595244001096
8 0.051214993207654394
9 0.04225069540054605
10 0.035754590114306165
11 0.030856260180445277
12 0.027046391064029753
13 0.024008419696393744
14 0.021535955106291943
15 0.019489090476991107
16 0.017769862635450603
17 0.016307771593137333
18 0.015050875844770259
19 0.01396012018370537


In [12]:
# accessing gradients
n.layers[0].neurons[0].w[0].grad

0.008272943016055311